# Data Read

```python
import pandas as pd

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

display(train)
display(test)

# data 정리 

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

train_x = train.drop(columns=['Prospect', 'ID'])
test_x = test.drop(columns=['ID'])

train_y = train['Prospect']

# 스케일링(표준화)
scaler = StandardScaler()
train_x_scaled = scaler.fit_transform(train_x)
test_x_scaled = scaler.transform(test_x)

# 스케일링된 데이터프레임 생성
columns = train_x.columns
train_x = pd.DataFrame(train_x_scaled, columns=columns)
test_x = pd.DataFrame(test_x_scaled, columns=columns)

train_x = sm.add_constant(train_x) # 상수항 추가
test_x = sm.add_constant(test_x) # 상수항 추가
train_x


```

## 데이터 검증 준비 

```python
train_x, val_x, train_y, val_y = train_test_split(train_x, train_y, test_size=0.2, random_state=42)
```

## 모델 학습 

```python
model = sm.Logit(train_y, train_x)
result = model.fit()
print(result.summary())
```

# 모델 요약 

```
                           Logit Regression Results                           
==============================================================================
Dep. Variable:               Prospect   No. Observations:                 2415
Model:                          Logit   Df Residuals:                     2407
Method:                           MLE   Df Model:                            7
Date:                Tue, 12 May 2026   Pseudo R-squ.:                  0.3045
Time:                        13:31:07   Log-Likelihood:                -1098.2
converged:                       True   LL-Null:                       -1578.9
Covariance Type:            nonrobust   LLR p-value:                2.529e-203
=================================================================================
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const            -0.7804      0.055    -14.273      0.000      -0.888      -0.673
Age              -1.5943      0.068    -23.577      0.000      -1.727      -1.462
Height            0.0269      0.081      0.334      0.738      -0.131       0.185
Weight           -0.0281      0.080     -0.352      0.725      -0.184       0.128
PaceTotal         0.0086      0.064      0.133      0.894      -0.118       0.135
ShootingTotal    -0.0064      0.069     -0.093      0.926      -0.141       0.129
PassingTotal     -0.0153      0.073     -0.211      0.833      -0.157       0.127
Composure        -0.1929      0.065     -2.990      0.003      -0.319      -0.066
=================================================================================
```

## 예측 결과 (Confusion_matrix)

|-|Predicted Negative|Predicted Positive|
|---|---|---|
|Actual Negative|TN|FP|
|Actual Positive|FN|TP|

```python
from sklearn.metrics import confusion_matrix

pred = result.predict(val_x)
pred = (pred > 0.5).astype(int)

conf_matrix = confusion_matrix(val_y, pred)
conf_matrix
```

```
array([[319,  66],
       [ 88, 131]])
```

## 모델 평가 

```python
from sklearn.metrics import classification_report

print(classification_report(val_y, pred))
```

```
              precision    recall  f1-score   support

           0       0.78      0.83      0.81       385
           1       0.66      0.60      0.63       219

    accuracy                           0.75       604
   macro avg       0.72      0.71      0.72       604
weighted avg       0.74      0.75      0.74       604
```

## 예측

```python
pred = result.predict(test_x)
pred = (pred > 0.5).astype(int)
pred
```

# 분류 모델 성능 평가 지표 (정확도, 정밀도, 재현율)
혼동 행렬(Confusion Matrix)을 바탕으로 분류 모델의 예측 성능을 측정하는 3가지 핵심 수식에 대한 설명입니다.

## 예시 파일
[Scikit-learn 분류 평가지표 공식 가이드](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics)

## 답변
수식을 이해하기 위해 먼저 혼동 행렬의 4가지 요소를 알아야 합니다.
- **TP (True Positive)**: 실제 양성(1)을 양성(1)으로 올바르게 예측
- **TN (True Negative)**: 실제 음성(0)을 음성(0)으로 올바르게 예측
- **FP (False Positive)**: 실제 음성(0)인데 양성(1)으로 잘못 예측 (1형 오류)
- **FN (False Negative)**: 실제 양성(1)인데 음성(0)으로 잘못 예측 (2형 오류)

이를 바탕으로 지표를 계산합니다.

### 1. 정확도 (Accuracy)
**"전체 예측 중에서 정답을 맞춘 비율"**
$$
\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
$$
- 가장 직관적인 평가 지표입니다.
- 단점: 정상 데이터 99개, 불량 데이터 1개일 때 모두 '정상'으로만 찍어도 99%가 나오므로 데이터 불균형이 심할 때는 왜곡될 수 있습니다.

### 2. 정밀도 (Precision)
**"모델이 양성(1)이라고 예측한 것 중 진짜 양성(1)인 비율"**
$$
\text{Precision} = \frac{TP}{TP + FP}
$$
- 모델 예측의 확실성을 나타냅니다.
- **언제 중요한가?** 쓸데없는 오작동(FP)을 줄여야 할 때. (예: 스팸 메일 필터 - 정상 메일을 스팸으로 지우면 큰일남)

### 3. 재현율 (Recall)
**"실제 양성(1)인 정답 데이터 중에서 모델이 맞춘 비율"**
$$
\text{Recall} = \frac{TP}{TP + FN}
$$
- 실제 정답을 놓치지 않고 얼마나 잘 찾아내는지를 나타냅니다. (민감도 또는 TPR이라고도 부름)
- **언제 중요한가?** 정답을 놓치는 것(FN)이 치명적일 때. (예: 암 환자 진단 - 암 환자를 정상으로 오진해서 돌려보내면 치명적임)

### 추가 자료
- [Scikit-learn metrics.classification_report 공식 문서](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html)
- [위키백과: 정밀도와 재현율 (Precision and recall)](https://ko.wikipedia.org/wiki/%EC%A0%95%EB%B0%80%EB%8F%84%EC%99%80_%EC%9E%AC%ED%98%84%EC%9C%A8)


# fit_transform, transform 및 add_constant 함수의 역할
데이터 전처리와 statsmodels 회귀 분석을 위해 사용하는 함수들에 대한 간단한 설명입니다.

## 예시 파일
[scikit-learn 데이터 전처리 공식 문서](https://scikit-learn.org/stable/modules/preprocessing.html)

주어진 코드에서 각 함수는 다음 역할을 수행합니다.

1. **`fit_transform(train_x)`**
   - **역할**: 훈련 데이터(`train_x`)의 평균값과 표준편차를 먼저 **학습(fit)**한 다음, 그 기준에 맞춰 데이터를 스케일링으로 **변환(transform)**합니다.
   - **주의**: 기준(평균, 편차)을 세우는 작업이 포함되므로 반드시 훈련 데이터(Train data)에만 사용해야 합니다.

2. **`transform(test_x)`**
   - **역할**: 위 `fit_transform` 단계에서 훈련 데이터를 통해 얻은 '평균'과 '표준편차' 기준을 그대로 가져와서 테스트 데이터(`test_x`)를 **변환만** 합니다.
   - **이유**: 평가 데이터(Test)가 훈련 데이터(Train)와 완전히 동일한 잣대로 평가받도록 하기 위함입니다. 테스트 데이터에서 새롭게 기준을 세우면(fit) 절대 안 됩니다.

3. **`add_constant()`**
   - **역할**: 기존 데이터프레임에 모든 값이 1인 상수항(Constant) 열을 추가합니다. 
   - **이유**: `statsmodels`의 회귀 모델(Logit, OLS 등)은 `scikit-learn`과 다르게 $y = wx + b$ 식에서 절편($b$, Bias)을 자동으로 만들어 주지 않습니다. 따라서 이 함수를 통해 강제로 절편 칸(상수 1)을 만들어 주어야 모델이 정상적으로 절편을 계산합니다.

### 추가 자료
- [Scikit-Learn: StandardScaler.fit_transform 공식 문서](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html#sklearn.preprocessing.StandardScaler.fit_transform)
- [Scikit-Learn: StandardScaler.transform 공식 문서](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html#sklearn.preprocessing.StandardScaler.transform)
- [Statsmodels: add_constant 공식 문서](https://www.statsmodels.org/stable/generated/statsmodels.tools.tools.add_constant.html)